# JFDS Experiment — 01: Theory & Research Hypothesis

**Part of a 5-notebook series** (see `README.md`). This notebook is self-contained — it needs no
trained models or dataset — and does two things a reviewer will look for before trusting any
number in `02`–`05`:

1. States the research hypotheses being tested, precisely enough to be falsifiable.
2. Derives *why* the JFDS score equation and its Adaptive-λ schedule take the specific functional
   forms they do, rather than presenting them as arbitrary design choices.

Two small illustrative figures below (synthetic data, not MovieLens) support the derivations.

---
## 1. Research Hypotheses

JFDS (Joint Fairness-Diversity Score) is a re-ranking layer that combines relevance, exposure
fairness (across popularity tiers), and intra-list diversity into a single greedy scoring
function. The experiments in this project are designed to test three falsifiable hypotheses.

**H1 (Joint dominance).** *A jointly-optimized fairness-diversity re-ranker (JFDS) achieves higher
fairness **and** higher diversity than a relevance-only baseline (Top-K), and does not lose
materially more NDCG than either of the two single-objective specialists (MMR for diversity,
Fairness-only for fairness) that each optimize just one of those two dimensions.*

- *Falsified if:* JFDS's NDCG loss relative to Top-K is not offset by fairness+diversity gains
  that a single-objective specialist could not also deliver at a comparable NDCG cost — i.e. if
  JFDS is simply dominated by MMR or Fairness-only on the objective(s) that method was designed
  for, at equal or lower relevance cost.
- *Tested in:* `02_main_experiments` (headline comparison), `04_significance_and_robustness`
  (paired significance tests, bootstrap CIs, group-fairness breakdown, aggregate-diversity /
  exposure equity).

**H2 (Squared-gap fairness term).** *A squared relative-deficit fairness penalty,
`(gap/target)²`, produces a more self-limiting correction than a linear penalty `gap/target` —
specifically, it contributes less marginal fairness pressure once a tier is close to its target
share, and more once a tier is far from it — and this functional-form choice measurably affects
the fairness/diversity/utility trade-off, not just its magnitude.*

- *Falsified if:* replacing the squared gap with a linear gap (holding λ_f, λ_d fixed) produces no
  significant difference in fairness (F), diversity (D), or utility (NDCG).
- *Tested in:* `03_ablation_and_complexity` (linear-gap ablation arm).

**H3 (Position-adaptive λ).** *Because DCG's rank discount `1/log2(rank+2)` shrinks with rank,
concentrating fairness/diversity pressure at later list positions (Adaptive-λ) achieves fairness
and diversity gains comparable to a static-λ JFDS at a lower NDCG cost, because the positions where
fairness/diversity pressure is heaviest are also the positions where a relevance sacrifice is
cheapest in DCG terms.*

- *Falsified if:* Adaptive-λ JFDS does not improve NDCG relative to static-λ JFDS at matched (or
  better) fairness/diversity levels — i.e. if the schedule's placement of pressure has no
  measurable relationship to the DCG discount curve.
- *Tested in:* `04_significance_and_robustness` (`AdaptiveJFDS` vs. static `JFDS`),
  `05_appendix_supplementary` (schedule-shape sensitivity — is H3's benefit specific to the
  particular linear schedule chosen, or does it hold across schedule shapes).

---
## 2. Problem Setup: Multi-Objective Re-Ranking as Scalarization

For a user $u$ with a candidate pool $C_u$ (size $P$) produced by a base recommender (SVD or BPR),
re-ranking selects an ordered list $S \subset C_u$, $|S|=k$, to jointly optimize three competing
objectives:

- **Relevance** $U(S)$ — how well $S$ matches the user's preferences (measured downstream by NDCG).
- **Fairness** $F(S)$ — how evenly exposure is distributed across popularity tiers (head/mid/tail).
- **Diversity** $D(S)$ — how dissimilar the items in $S$ are from each other (genre-based ILD).

These three objectives are **not simultaneously maximizable** in general: the most relevant items
for most users skew toward popular ("head") content (relevance vs. fairness tension), and the most
relevant items for a given user are often genre-similar to each other (relevance vs. diversity
tension). This is a textbook multi-objective optimization problem, and JFDS's score function is a
**linear scalarization** of it — a single scalar objective built from a convex combination of the
three:

$$\text{score}(i \mid S) = \lambda_u \cdot rel(i) + \lambda_f \cdot \text{fair\_boost}(i \mid S)
+ \lambda_d \cdot \text{diversity}(i, S), \qquad \lambda_u + \lambda_f + \lambda_d = 1,\;\; \lambda_\bullet \ge 0$$

**Why linear scalarization, and what it can and cannot do.** A classical result in multi-objective
optimization is that for a *convex* feasible region, sweeping the weights $(\lambda_u, \lambda_f,
\lambda_d)$ over the simplex traces out exactly the Pareto-optimal frontier — every point on the
frontier corresponds to *some* choice of weights, and every choice of weights lands on the
frontier (assuming each per-item term is bounded, which fairness gap and mean-similarity are, both
$\in[0,1]$ by construction). The caveat: if the achievable $(U,F,D)$ region has a **non-convex** or
concave section, linear scalarization can never reach solutions there, no matter how the weights
are tuned — only a *convex combination of two other reachable points* is representable, not
solutions on a concave part of the boundary. The `01`-adjacent Pareto-frontier analysis in
`05_appendix_supplementary` (R1) gives an empirical first look at how much of the achievable region
is convex versus not, for this dataset.

This is also why JFDS is compared against MMR and Fairness-only rather than only against Top-K:
those two baselines are the $\lambda_d{=}0$ and $\lambda_f{=}0$ **corners** of the same simplex.
If JFDS (an interior point of the simplex) cannot beat both corner solutions on the dimension each
corner was chosen to maximize, jointly optimizing was not worth doing — this is exactly H1.

---
## 3. Why `fair_boost` Uses a Squared Relative Gap

For a candidate item $i$ belonging to popularity tier $t(i) \in \{\text{head}, \text{mid}, \text{tail}\}$,
after $n$ items have already been selected into $S$:

$$\text{current\_share}(t) = \frac{|\{j \in S : t(j)=t\}|}{n}, \qquad
\text{gap}(t) = \max\!\big(0,\; \text{target}(t) - \text{current\_share}(t)\big)$$

$$\text{fair\_boost}(i \mid S) = \left(\frac{\text{gap}(t(i))}{\text{target}(t(i))}\right)^{2}$$

Two design choices need justifying: **(a)** squaring the gap rather than using it linearly, and
**(b)** normalizing by `target(t)` rather than using the raw gap.

**(a) Squaring.** A linear penalty (`gap/target`) applies the *same marginal boost per unit of
deficit* everywhere — an item from a tier that is 5% below target gets exactly half the boost of
an item from a tier 10% below target. A **squared** penalty is convex in the gap: it is small when
the tier is nearly at its target (self-limiting — avoids overcorrecting once fairness is roughly
satisfied) and grows superlinearly as the deficit widens (aggressive — prioritizes tiers that are
badly under-represented over tiers that are only slightly under-represented). This is the same
shape as a quadratic penalty term in a soft-constraint / penalty-method formulation of constrained
optimization: it enforces the constraint ($\text{current\_share} \to \text{target}$) increasingly
strongly as the violation grows, while not fighting for the last few percentage points as hard as
it fights for the first big ones. **H2** tests whether this functional-form choice is actually
load-bearing, not just aesthetically preferable — see `03_ablation_and_complexity`.

**(b) Normalizing by target.** Dividing by `target(t)` converts an absolute percentage-point gap
into a *relative* deficit. Without this, a tier with a smaller target share (e.g. if targets were
unequal, unlike the equal 1/3 targets used here) would structurally receive a smaller raw gap and
therefore systematically less correction even at the same *relative* level of under-representation
— normalization keeps `fair_boost` comparable across tiers regardless of their target size, which
matters if this equation is ever extended beyond the current equal-thirds tier targets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Illustrative only -- synthetic gap values, not MovieLens data.
gap_frac = np.linspace(0, 1, 200)  # gap / target, i.e. relative deficit
linear_penalty  = gap_frac
squared_penalty = gap_frac ** 2

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(gap_frac, linear_penalty,  label='linear:  gap / target', linewidth=2)
ax.plot(gap_frac, squared_penalty, label='squared: (gap / target)^2  [used by JFDS]', linewidth=2)
ax.fill_between(gap_frac, squared_penalty, linear_penalty, alpha=0.15, color='grey',
                 label='region where squared < linear\n(self-limiting near target)')
ax.set_xlabel('Relative deficit  gap / target  (0 = at target, 1 = maximally under-represented)')
ax.set_ylabel('fair_boost contribution')
ax.set_title('Linear vs. squared fairness penalty shape')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fairness_penalty_shapes.png', dpi=110, bbox_inches='tight')
plt.show()

print('At 20% relative deficit: linear =', round(0.2, 3), ' squared =', round(0.2 ** 2, 3))
print('At 80% relative deficit: linear =', round(0.8, 3), ' squared =', round(0.8 ** 2, 3))
print('The squared penalty is ~4x smaller near the target and starts catching up to (then '
      'exceeding, in relative terms) the linear penalty as the deficit grows -- exactly the '
      'self-limiting-then-aggressive shape argued for above.')


---
## 4. Why the Diversity Term Uses Mean Similarity, Not Max Similarity

$$\text{diversity}(i, S) = 1 - \frac{1}{|S|}\sum_{j \in S} \text{sim}(i, j)
\qquad \text{(JFDS, mean-based)}$$

$$\text{mmr\_max\_sim}(i, S) = \max_{j \in S}\text{sim}(i, j)
\qquad \text{(MMR baseline, max-based, Carbonell \& Goldstein 1998)}$$

Both are legitimate diversification objectives, but they optimize different things. MMR's
max-similarity term is a **worst-case** (facility-location-style) objective: it only cares about
the single most similar already-selected item, so it aggressively avoids near-duplicates but is
indifferent to overall genre spread once no single close duplicate exists. JFDS's mean-similarity
term is an **average-case** objective: every already-selected item contributes to the penalty, so
it continues to push toward broader genre coverage even after the worst-case duplicate risk is
gone.

The practical reason to prefer the mean-based formulation here specifically: **the evaluation
metric ILD (intra-list diversity) used everywhere in `02`–`05` is itself defined as
`1 − mean pairwise similarity`** — the same averaging as JFDS's diversity term, not MMR's max. This
means JFDS's optimization objective and its evaluation metric are aligned by construction, while
MMR is being evaluated on a metric (mean pairwise dissimilarity) that is a good proxy for, but not
identical to, what it was actually built to optimize (worst-case dissimilarity). This is disclosed
explicitly here — and is precisely *why* MMR is retained as a baseline rather than dropped: seeing
whether JFDS's metric-aligned diversity term outperforms MMR's metric-adjacent one, on the metric
both are measured by, is itself informative and is exactly the diversity-specific half of H1.

---
## 5. Why Adaptive-λ Grows With List Position

Static JFDS uses one $(\lambda_f, \lambda_d)$ pair for every position $k$ in the list. Adaptive-λ
JFDS instead grows both weights with list position:

$$t_k = \frac{k}{K-1}, \qquad
\lambda_f(k) = \lambda_f^{start} + (\lambda_f^{end}-\lambda_f^{start})\cdot t_k^{\,p}, \qquad
\lambda_d(k) = \lambda_d^{start} + (\lambda_d^{end}-\lambda_d^{start})\cdot t_k^{\,p}$$

**The theoretical motivation is a mismatch between where JFDS spends its fairness/diversity budget
and where NDCG is most sensitive to a relevance sacrifice.** NDCG discounts each list position by
$1/\log_2(\text{rank}+2)$ — position 1 is weighted $1/\log_2(3) \approx 0.63$, while position 10 is
weighted $1/\log_2(11) \approx 0.29$, roughly **half** as much. A static-λ JFDS spends the *same*
fairness/diversity pressure at position 1 (where a relevance sacrifice costs the most NDCG) as at
position 10 (where it costs roughly half as much). Shifting the schedule so that
$\lambda_f, \lambda_d$ start small (utility-heavy, protecting the expensive early ranks) and grow
toward the end of the list (where the same absolute drop in relevance costs less NDCG) should, in
principle, buy the same total fairness/diversity gain at a lower NDCG cost — this is H3, and it is
directly testable because both the static and adaptive variants can be evaluated on identical
candidate pools and identical fairness/diversity targets.

$p$ controls the shape of the ramp: $p{=}1$ is linear, $p<1$ front-loads the increase (reaches high
fairness/diversity pressure quickly, then plateaus), $p>1$ back-loads it (stays utility-heavy for
longer, then ramps sharply near the end of the list). The schedule-shape sensitivity analysis in
`05_appendix_supplementary` checks whether H3's conclusion depends on this shape choice or holds
across it.

In [ ]:
# Illustrative only: NDCG rank discount vs. an example Adaptive-lambda schedule (synthetic, K=10).
K_illustrative = 10
ranks = np.arange(K_illustrative)
dcg_discount = 1.0 / np.log2(ranks + 2)
dcg_discount_norm = dcg_discount / dcg_discount.max()  # normalize to [~0.46, 1] for overlay

def schedule_lambdas(step, k_total, lf_start, lf_end, ld_start, ld_end, p=1.0):
    t = step / max(1, k_total - 1)
    lam_f = lf_start + (lf_end - lf_start) * (t ** p)
    lam_d = ld_start + (ld_end - ld_start) * (t ** p)
    return lam_f, lam_d

DEFAULT_SCHEDULE = dict(lf_start=0.05, lf_end=0.45, ld_start=0.05, ld_end=0.35, p=1.0)
lam_f_curve, lam_d_curve = zip(*[schedule_lambdas(s, K_illustrative, **DEFAULT_SCHEDULE) for s in ranks])

fig, ax1 = plt.subplots(figsize=(8, 5.5))
ax1.plot(ranks + 1, dcg_discount, color='black', linewidth=2, marker='o',
         label='NDCG rank discount  1/log2(rank+2)')
ax1.set_xlabel('List position (rank)')
ax1.set_ylabel('NDCG discount weight', color='black')
ax1.set_xticks(ranks + 1)

ax2 = ax1.twinx()
ax2.plot(ranks + 1, lam_f_curve, color='#E67E22', linewidth=2, marker='s', label='lambda_f(k)')
ax2.plot(ranks + 1, lam_d_curve, color='#2980B9', linewidth=2, marker='^', label='lambda_d(k)')
ax2.set_ylabel('Adaptive lambda weight')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='center right')
ax1.set_title('NDCG discount falls as lambda_f, lambda_d rise -- the motivation for Adaptive-lambda')
plt.tight_layout()
plt.savefig('adaptive_schedule_vs_dcg_discount.png', dpi=110, bbox_inches='tight')
plt.show()

print(f"Position 1 NDCG weight: {dcg_discount[0]:.3f}  |  Position {K_illustrative} NDCG weight: {dcg_discount[-1]:.3f}"
      f"  (ratio: {dcg_discount[0]/dcg_discount[-1]:.2f}x)")
print(f"lambda_f grows from {lam_f_curve[0]:.2f} to {lam_f_curve[-1]:.2f} over the same {K_illustrative} positions.")


---
## Summary — What Each Notebook Validates

| Hypothesis | Where tested | Falsification condition |
|---|---|---|
| H1 — Joint dominance | `02`, `04` | JFDS dominated by a single-objective specialist on its own metric at equal/lower NDCG cost |
| H2 — Squared-gap fairness term | `03` | No significant F/D/NDCG difference vs. a linear-gap ablation |
| H3 — Position-adaptive λ | `04`, `05` | No NDCG advantage for Adaptive-λ at matched fairness/diversity, or advantage vanishes across schedule shapes |

This notebook makes no empirical claims of its own — everything above is a derivation and a
falsifiable prediction. `02` onward is where those predictions are actually tested against
MovieLens 1M.